In [1]:
import torch
from torch import nn
from torch.nn import functional as F

In [2]:
class Residual(nn.Module):
    def __init__(self, num_channels, use_1x1conv=False, strides=1): 
        super().__init__()
        self.conv1 = nn.LazyConv2d(num_channels, kernel_size=3, padding=1, 
                                   stride=strides)
        self.conv2 = nn.LazyConv2d(num_channels, kernel_size=3, padding=1)
        if use_1x1conv: 
            self.conv3 = nn.LazyConv2d(num_channels, kernel_size=1, 
                                       stride=strides)
        else: 
            self.conv3 = None
        self.bn1 = nn.LazyBatchNorm2d()
        self.bn2 = nn.LazyBatchNorm2d()

    def forward(self, X): 
        Y = F.relu(self.bn1(self.conv1(X)))
        Y = self.bn2(self.conv2(Y))
        if self.conv3: 
            X = self.conv3(X)
        Y += X
        return F.relu(Y)

In [7]:
class ResNet(nn.Module): 
    def __init__(self):
        super().__init__()
        # block 1
        self.b1 = nn.Sequential(
            nn.LazyConv2d(64, kernel_size=7, stride=2, padding=3), 
            nn.LazyBatchNorm2d(), 
            nn.ReLU(), 
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )
        self.b2 = self.block(2, 64, first_block=True)
        self.b3 = self.block(2, 128)
        self.b4 = self.block(2, 256)
        self.b5 = self.block(2, 512)

        self.final = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)), 
            nn.Flatten(), 
            nn.LazyLinear(10)
        )

    def block(self, num_residuals, num_channels, first_block=False): 
        blk = []
        for i in range(num_residuals): 
            if i == 0 and not first_block: 
                blk.append(Residual(num_channels, use_1x1conv=True, strides=2))
            else: 
                blk.append(Residual(num_channels))
        return nn.Sequential(*blk)

    def forward(self, X): 
        X = self.b1(X)
        X = self.b2(X)
        X = self.b3(X)
        X = self.b4(X)
        X = self.b5(X)

        X = self.final(X)
        return X

In [4]:
# blk = Residual(3)
# X = torch.randn(4, 3, 6, 6)
# blk(X).shape

In [5]:
# blk = Residual(6, use_1x1conv=True, strides=2)
# blk(X).shape

In [6]:
X = torch.randn(1, 3, 224, 224)

net = ResNet()
Y = net(X)
Y.shape

torch.Size([1, 512, 7, 7])

In [8]:
net = ResNet()
net(X).shape

torch.Size([1, 10])